# 01 - Schema Inspection & Ingestion

**Tujuan notebook ini:**

1. Scan folder `dataset/` dan kelompokkan 486 file CSV NBA/WNBA (play-by-play & shot detail, 1996-2024) berdasarkan *family* (nama dataset), musim (season), dan jenis game (regular/playoff).
2. Inspeksi skema tiap family lewat 1 file sampel: cek nama kolom, tipe data, dan jumlah baris.
3. Ingest penuh: baca semua file tiap family, tandai `season` & `game_type`, lalu union jadi satu DataFrame per family.
4. Sanity check: total baris & null count per kolom, serta deteksi apakah skema konsisten antar tahun dalam 1 family yang sama.
5. Simpan hasil ingestion sebagai Parquet (partitioned by `season`) ke `output/parquet/{family}/` untuk dipakai di notebook EDA/analisis selanjutnya.

Semua logic reusable ditaruh di `src/utils.py` (file discovery) dan `src/ingestion.py` (Spark pipeline), notebook ini hanya memanggil dan menampilkan hasilnya.

## Setup: SparkSession & environment

Catatan environment (Windows): PySpark butuh Java 8/11/17 (Java versi terbaru seperti 21+/23 tidak didukung Spark 3.5.x dan akan gagal start), serta `winutils.exe` + `hadoop.dll` untuk operasi filesystem lokal (termasuk menulis Parquet). Env var di bawah ini hanya diisi kalau belum di-set di sistem, supaya notebook tetap portable.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Java/Hadoop setup for local Windows runs (Spark 3.5.x needs Java 8/11/17, not 21+/23).
os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Java\jre-1.8")
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

# Driver memory MUST be set before the JVM gateway launches (SparkSession.builder.config()
# is too late in local mode - the JVM is already up by then), so this goes through
# PYSPARK_SUBMIT_ARGS instead. Dataset is ~13GB of CSV, so give the single local-mode
# JVM (which doubles as driver+executor) plenty of headroom.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g pyspark-shell"

DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
PARQUET_DIR = os.path.join(OUTPUT_DIR, "parquet")
SCHEMA_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "schema_summary.json")

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("NBA-WNBA-PlayByPlay-Ingestion")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## Section 1: Dataset discovery

Scan `dataset/` dan kelompokkan file berdasarkan family (misal `nbastats`, `shotdetail`, `cdnnba`, dst), lalu tampilkan berapa file & rentang musim yang tersedia per family.

In [3]:
import pandas as pd

from src.utils import scan_dataset, summarize_families

families = scan_dataset(DATASET_DIR)
summary = summarize_families(families)

print(f"Jumlah family ditemukan: {len(families)}")
pd.DataFrame(summary).drop(columns=["seasons"])

Jumlah family ditemukan: 2


,family,total_files,regular_files,playoff_files,season_min,season_max
0,nbastats,58,29,29,1996,2024
1,shotdetail,58,29,29,1996,2024


## Section 2: Schema inspection

Untuk tiap family, load 1 file sampel dengan PySpark lalu tampilkan `printSchema()`, `show(5)`, dan row count file sampel tersebut. Hasil ringkasan (kolom + tipe + jumlah baris) disimpan ke `output/schema_summary.json`.

In [4]:
from src.ingestion import inspect_schema, save_schema_summary

schema_summaries = []
for family, files in sorted(families.items()):
    schema_summaries.append(inspect_schema(spark, family, files))

save_schema_summary(schema_summaries, SCHEMA_SUMMARY_PATH)
print(f"\nSchema summary disimpan ke {SCHEMA_SUMMARY_PATH}")


=== Family: nbastats (sample: nbastats_po_1996.csv) ===
root
 |-- GAME_ID: integer (nullable = true)
 |-- EVENTNUM: integer (nullable = true)
 |-- EVENTMSGTYPE: integer (nullable = true)
 |-- EVENTMSGACTIONTYPE: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- WCTIMESTRING: string (nullable = true)
 |-- PCTIMESTRING: timestamp (nullable = true)
 |-- HOMEDESCRIPTION: string (nullable = true)
 |-- NEUTRALDESCRIPTION: string (nullable = true)
 |-- VISITORDESCRIPTION: string (nullable = true)
 |-- SCORE: string (nullable = true)
 |-- SCOREMARGIN: string (nullable = true)
 |-- PERSON1TYPE: integer (nullable = true)
 |-- PLAYER1_ID: integer (nullable = true)
 |-- PLAYER1_NAME: string (nullable = true)
 |-- PLAYER1_TEAM_ID: integer (nullable = true)
 |-- PLAYER1_TEAM_CITY: string (nullable = true)
 |-- PLAYER1_TEAM_NICKNAME: string (nullable = true)
 |-- PLAYER1_TEAM_ABBREVIATION: string (nullable = true)
 |-- PERSON2TYPE: integer (nullable = true)
 |-- PLAYER2_ID: integ

## Section 3: Full ingestion

Baca SEMUA file tiap family (bukan cuma sampel), tambahkan kolom `season` & `game_type` dari nama file, lalu union jadi satu DataFrame per family. File dibaca satu per satu (bukan 1 wildcard glob) karena beberapa family (misal `cdnnba`, `datanba`) ternyata skemanya berubah antar tahun - `unionByName(allowMissingColumns=True)` dipakai supaya kolom tetap sejajar berdasarkan nama, bukan posisi.

In [5]:
from src.ingestion import ingest_all

dataframes, schema_reports = ingest_all(spark, families)

row_counts = []
for family, df in sorted(dataframes.items()):
    row_counts.append({"family": family, "union_row_count": df.count()})

pd.DataFrame(row_counts)

,family,union_row_count
0,nbastats,16699012
1,shotdetail,6094438


In [6]:
# Konsistensi skema antar file dalam 1 family yang sama
pd.DataFrame(schema_reports)

,family,num_files,distinct_schema_variants,schema_consistent
0,nbastats,58,1,True
1,shotdetail,58,1,True


## Section 4: Sanity check

Hitung total baris dan null count per kolom untuk tiap DataFrame hasil ingestion, supaya ketahuan kolom mana yang banyak kosong atau ada anomali.

In [7]:
from src.ingestion import sanity_check

sanity_results = {}
for family, df in sorted(dataframes.items()):
    result = sanity_check(family, df)
    sanity_results[family] = result
    print(f"\n=== {family} | total_rows={result['total_rows']} ===")
    null_df = pd.DataFrame(
        sorted(result["null_counts"].items(), key=lambda x: -x[1]),
        columns=["column", "null_count"],
    )
    null_df["null_pct"] = (null_df["null_count"] / result["total_rows"] * 100).round(2)
    display(null_df)


=== nbastats | total_rows=16699012 ===


,column,null_count,null_pct
0,NEUTRALDESCRIPTION,16305822,97.65
1,PLAYER3_NAME,16277955,97.48
2,PLAYER3_TEAM_ID,16272630,97.45
3,PLAYER3_TEAM_CITY,16272630,97.45
4,PLAYER3_TEAM_NICKNAME,16272630,97.45
5,PLAYER3_TEAM_ABBREVIATION,16272630,97.45
6,SCORE,12306848,73.70
7,SCOREMARGIN,12306848,73.70
8,PLAYER2_NAME,11843236,70.92
9,PLAYER2_TEAM_ID,11818294,70.77



=== shotdetail | total_rows=6094438 ===


,column,null_count,null_pct
0,SHOT_TYPE,1,0.0
1,GRID_TYPE,0,0.0
2,GAME_ID,0,0.0
3,GAME_EVENT_ID,0,0.0
4,PLAYER_ID,0,0.0
5,PLAYER_NAME,0,0.0
6,TEAM_ID,0,0.0
7,TEAM_NAME,0,0.0
8,PERIOD,0,0.0
9,MINUTES_REMAINING,0,0.0


## Section 5: Save to Parquet

Tulis tiap DataFrame family ke `output/parquet/{family}/`, dipartisi berdasarkan kolom `season`, supaya bisa dibaca efisien di notebook EDA selanjutnya tanpa parsing ulang CSV.

In [8]:
from src.ingestion import save_family_parquet

for family, df in sorted(dataframes.items()):
    path = save_family_parquet(df, family, PARQUET_DIR)
    print(f"Saved {family} -> {path}")

Saved nbastats -> e:\Kuliah\BigDat\KelompokN1\output\parquet\nbastats
Saved shotdetail -> e:\Kuliah\BigDat\KelompokN1\output\parquet\shotdetail


In [9]:
spark.stop()